#### Multiple Lights and Shadows

##### Objective: Using the Phong-Reflectance models from multiple areas to create multiple points of lights and understand how to deal with shadows.

We will revert to the Initial version of the reflectance model to model light intensity for the sake of simplicity (but indeed this is three parts of the R, G, B): $$I_{refl} = I_E + K_AI_{AMB} + K_DI_L(\vec{N} \cdot \vec{L}) + K_SI_L(\vec{V} \cdot \vec{\hat{L}})^n $$

- $I_L$ - Is the intensity of the light arriving from the light source (Which has 3 components)
This parameter is supposed to encompass all light sources so we open this up as follows: 

- For every Light source $j$, we denote the light intensity $I_{Lj}$, reaching a point on the surface.

As such we'll obtain: 

$$
I_{\text{refl}} = I_E + K_A I_{\text{AMB}} + \sum_j \left( K_D I_{L_j} (\vec{N} \cdot \vec{L}_j) + K_S I_{L_j} (\vec{V} \cdot \vec{\hat{R}}_j)^n \right)
$$


#### Several Light sources and many object

Q. How do account for the shadow produce by some objects onto others from the reflection of the light source?

A. We ask ourselves which light sources are occluded from a perticular view point in the scene? If we're able to determine such a point then this point becomes a shadow.

- This question can be included in the equation above as follows:

$$
I_{\text{refl}} = I_E + K_A I_{\text{AMB}} + \sum_j s_j\left( K_D I_{L_j} (\vec{N} \cdot \vec{L}_j) + K_S I_{L_j} (\vec{V} \cdot \vec{\hat{R}}_j)^n \right)
$$

Where $s_j$ is an indicator if light $l_j$ is occluded or not (output is $1$ or $0$).

#### How do we determine if a point is occluded?

This will occur if the line of sight from a perticular light source isn't clear. This can be done be casting a ray from a light source, in which the closest object it intersects won't be occluded.

Note: So objects cast soft-shadows - Emits color of itself and also a shadow, to get a better understanding of the factor (as the indicator is binary) we'd cast 4 rays from an object/light source to the four courners of the object, this will produce $$s_j = \frac{\sum_i s_{ji}}{rays \ \ casted} \ \  where \ s_{j} \in [0,1] \  and \ s_{ji} \in \left\{0, 1 \right\} $$

#### The method used for this: 

```
Intersection FindIntersection (Ray ray (Shadow Rays), Scene scene)
{
    min_t = -nf
    min_primitive = NULL
    For each primitice in scene {
        t = Intersect(ray, primitive)
        if (t < min_t) {
            min_primitive = primitive
            min_t = t
        }
    }
    return Intersetion(min_t, min_primitive)
}
```

#### Summary on the GetColor method in Ray Casting

##### Reminder: At this stage of the algorithm we calculated the closest intersection point with scene objects

1. Claculate the colour intensity of the extended phong-Reflectance equation colour Component (RGB)

##### To perform this calculation we need: 
- The 13 parameters
    - $I_{Ei}, K_{Ai}, K_{Di}, K_{Si} \ and \ n \ where \ i \in \left \{R, G, B \right\}$
- $I_{Ai} \ \ where \ i \in \left \{R, G, B \right\}$ - The environment Light color
- $I_{L{j}i} \ where \ i \in \left \{R, G, B \right\}$ - The intensity of light coming from light source j.
    - Where $I_L$ is determined by the type of light (Directional, Point, Spot) thus having their own equation for $I_L$
- $s_j$ - The shadow factor for light source j, by shooting shadow rays to the light source from the 3D intersection point (Calculated in the previouos step of Algorithm).
- $\vec{N} \cdot \vec{L_j}$ - The cosine of the angfle between the direction to the lifght i and the normal. 
- $\vec{V} \cdot \vec{\hat{L_j}}$ - The cosine of the angle between the direction to the viewer V and teh mirroing of the vector $L_j$ to the light source j.

```
RGB GetColor(Scene scene, Ray ray, Point hit){
    //Ambiant and Emission Calculations
    RGB color = calcEmissionColor(scene) + calcAmbientColor(scene);
    // Diffuse & Specular Calculation
    for (int j = 0; j < getNumLights(scene;) j++){
        Light light = getLight(j, scene);
        float sj = calcShadowFactor(scene, hit);
        color = color + sj*calcDiffuseColor(scene, hit, ray, ligth)
            + sj*calcSpecularColor(scene,hit,ray,light);
    }
    return color;
}
```